## Step 1: Ingestion & Data Exploration — SALES_ANALYTICS_2025

**Project:** AI-Powered Sales Data Insights with Snowflake + Claude AI

**Architecture:** INGESTION → Transformation → Delivery → Observability

**What this notebook does:**
1. Connect to the existing `SALES_ANALYTICS_2025` database
2. Explore and profile all 8 raw tables
3. Identify data quality issues BEFORE transformation
4. Set up the `CURATED` schema for transformed data

Following [Snowflake DE_100 Quickstart](https://quickstarts.snowflake.com/guide/intro-to-data-engineering-python/) patterns using **pandas on Snowflake** (modin.pandas) and **Snowpark DataFrames**.

In [ ]:
# Snowpark Pandas API (same pattern as DE_100 notebook)
import modin.pandas as spd
import snowflake.snowpark.modin.plugin

# Snowpark DataFrame API
import snowflake.snowpark.functions as F
from snowflake.snowpark.context import get_active_session

In [ ]:
# Create a Snowpark session
session = get_active_session()

# Add query tag for troubleshooting and monitoring
session.query_tag = {
    "origin": "raj_manala",
    "name": "sales_analytics_ai_insights",
    "version": {"major": 1, "minor": 0},
    "attributes": {
        "is_project": 1,
        "source": "notebook",
        "step": "ingestion_exploration"
    }
}

print(f"Session created successfully!")
print(f"Current database: {session.get_current_database()}")
print(f"Current schema: {session.get_current_schema()}")
print(f"Current warehouse: {session.get_current_warehouse()}")

### Set up the database and schemas
We use `SALES_ANALYTICS_2025` database with two schemas:
- `RAW_DATA` — the 8 original dimension and fact tables (already loaded)
- `CURATED` — transformed analytical tables (we'll create this)

In [ ]:
-- Use the existing database
USE DATABASE SALES_ANALYTICS_2025;
USE SCHEMA RAW_DATA;

-- Create the CURATED schema for transformed data
CREATE SCHEMA IF NOT EXISTS CURATED;

-- Verify both schemas exist
SHOW SCHEMAS IN DATABASE SALES_ANALYTICS_2025;

### Explore the raw data tables
Let's see what we have in `RAW_DATA` schema — all 8 tables loaded from CSV files.

In [ ]:
-- List all tables in RAW_DATA
USE SCHEMA RAW_DATA;
SHOW TABLES IN SCHEMA RAW_DATA;

In [ ]:
-- Get row counts for all tables
SELECT 'DIM_CUSTOMER' AS table_name, COUNT(*) AS row_count FROM DIM_CUSTOMER
UNION ALL SELECT 'DIM_DATE', COUNT(*) FROM DIM_DATE
UNION ALL SELECT 'DIM_GEOGRAPHY', COUNT(*) FROM DIM_GEOGRAPHY
UNION ALL SELECT 'DIM_PRODUCT', COUNT(*) FROM DIM_PRODUCT
UNION ALL SELECT 'DIM_PRODUCT_CATEGORY', COUNT(*) FROM DIM_PRODUCT_CATEGORY
UNION ALL SELECT 'DIM_PRODUCT_SUBCATEGORY', COUNT(*) FROM DIM_PRODUCT_SUBCATEGORY
UNION ALL SELECT 'FACT_BUDGET', COUNT(*) FROM FACT_BUDGET
UNION ALL SELECT 'FACT_INTERNET_SALES', COUNT(*) FROM FACT_INTERNET_SALES
ORDER BY table_name;

### Explore FACT_INTERNET_SALES — the core sales table
This is our main fact table. Let's load it using **pandas on Snowflake** (same pattern as DE_100 notebook).

In [ ]:
# Load FACT_INTERNET_SALES using Snowpark DataFrame
fact_sales_sdf = session.table('FACT_INTERNET_SALES')

# Show the schema
print("FACT_INTERNET_SALES columns:")
for field in fact_sales_sdf.schema.fields:
    print(f"  {field.name:<30} {str(field.datatype):<20}")

print(f"\nTotal rows: {fact_sales_sdf.count()}")

In [ ]:
# Preview the data
fact_sales_sdf.show(5)

In [ ]:
# Key statistics on sales data
sales_stats = fact_sales_sdf.agg(
    F.count('*').alias('TOTAL_ORDERS'),
    F.sum('SALESAMOUNT').alias('TOTAL_REVENUE'),
    F.avg('SALESAMOUNT').alias('AVG_ORDER_VALUE'),
    F.min('ORDERDATE').alias('FIRST_ORDER'),
    F.max('ORDERDATE').alias('LAST_ORDER'),
    F.count_distinct('PRODUCTKEY').alias('UNIQUE_PRODUCTS'),
    F.count_distinct('CUSTOMERKEY').alias('UNIQUE_CUSTOMERS'),
    F.avg('UNITPRICE').alias('AVG_UNIT_PRICE'),
    F.sum('TAXAMT').alias('TOTAL_TAX'),
    F.sum('FREIGHT').alias('TOTAL_FREIGHT')
)

print("\n=== FACT_INTERNET_SALES Key Statistics ===")
sales_stats.show()

### Explore DIM_CUSTOMER — customer demographics

In [ ]:
# Load DIM_CUSTOMER using pandas on Snowflake (modin)
customer_mdf = spd.read_snowflake('DIM_CUSTOMER')

print(f"DIM_CUSTOMER: {len(customer_mdf)} rows × {customer_mdf.shape[1]} columns")
print(f"\nColumns: {list(customer_mdf.columns)}")
customer_mdf.head(5)

In [ ]:
# Customer demographics profile
print("=== Gender Distribution ===")
print(customer_mdf['GENDER'].value_counts())

print("\n=== Education Distribution ===")
print(customer_mdf['ENGLISHEDUCATION'].value_counts())

print("\n=== Occupation Distribution ===")
print(customer_mdf['ENGLISHOCCUPATION'].value_counts())

print("\n=== Marital Status ===")
print(customer_mdf['MARITALSTATUS'].value_counts())

print("\n=== Income Range ===")
print(f"  Min: ${customer_mdf['YEARLYINCOME'].min():,.0f}")
print(f"  Max: ${customer_mdf['YEARLYINCOME'].max():,.0f}")
print(f"  Mean: ${customer_mdf['YEARLYINCOME'].mean():,.0f}")

### Explore DIM_PRODUCT — product catalog

In [ ]:
# Load DIM_PRODUCT
product_mdf = spd.read_snowflake('DIM_PRODUCT')

print(f"DIM_PRODUCT: {len(product_mdf)} rows × {product_mdf.shape[1]} columns")
print(f"\nProduct key range: {product_mdf['PRODUCTKEY'].min()} to {product_mdf['PRODUCTKEY'].max()}")
product_mdf[['PRODUCTKEY', 'ENGLISHPRODUCTNAME', 'PRODUCTSUBCATEGORYKEY', 'STANDARDCOST', 'LISTPRICE', 'COLOR']].head(10)

In [ ]:
# Load product categories and subcategories
category_mdf = spd.read_snowflake('DIM_PRODUCT_CATEGORY')
subcategory_mdf = spd.read_snowflake('DIM_PRODUCT_SUBCATEGORY')

print("=== Product Categories ===")
print(category_mdf[['PRODUCTCATEGORYKEY', 'ENGLISHPRODUCTCATEGORYNAME']])

print("\n=== Product Subcategories (Bikes) ===")
bike_subcats = subcategory_mdf[subcategory_mdf['PRODUCTCATEGORYKEY'] == 1]
print(bike_subcats[['PRODUCTSUBCATEGORYKEY', 'ENGLISHPRODUCTSUBCATEGORYNAME']])

### DATA QUALITY CHECK #1: Referential Integrity
Do the keys in FACT_INTERNET_SALES match the dimension tables? This is critical — if keys don't match, our joins will lose data.

In [ ]:
-- CHECK: Do product keys in sales match DIM_PRODUCT?
-- This reveals the critical data quality issue!
SELECT 
    'Product keys in FACT_INTERNET_SALES' AS check_name,
    COUNT(DISTINCT PRODUCTKEY) AS unique_keys,
    MIN(PRODUCTKEY) AS min_key,
    MAX(PRODUCTKEY) AS max_key
FROM FACT_INTERNET_SALES

UNION ALL

SELECT 
    'Product keys in DIM_PRODUCT',
    COUNT(DISTINCT PRODUCTKEY),
    MIN(PRODUCTKEY),
    MAX(PRODUCTKEY)
FROM DIM_PRODUCT;

In [ ]:
-- Find which product keys in sales DON'T exist in DIM_PRODUCT
-- This is the gap we need to fix in Step 2!
SELECT DISTINCT f.PRODUCTKEY, 
    CASE WHEN p.PRODUCTKEY IS NULL THEN '❌ MISSING' ELSE '✓ Found' END AS status
FROM FACT_INTERNET_SALES f
LEFT JOIN DIM_PRODUCT p ON f.PRODUCTKEY = p.PRODUCTKEY
ORDER BY f.PRODUCTKEY;

In [ ]:
-- CHECK: Do customer keys in sales match DIM_CUSTOMER?
SELECT 
    COUNT(DISTINCT f.CUSTOMERKEY) AS sales_customers,
    COUNT(DISTINCT c.CUSTOMERKEY) AS dim_customers,
    COUNT(DISTINCT CASE WHEN c.CUSTOMERKEY IS NOT NULL THEN f.CUSTOMERKEY END) AS matched,
    COUNT(DISTINCT CASE WHEN c.CUSTOMERKEY IS NULL THEN f.CUSTOMERKEY END) AS unmatched
FROM FACT_INTERNET_SALES f
LEFT JOIN DIM_CUSTOMER c ON f.CUSTOMERKEY = c.CUSTOMERKEY;

### DATA QUALITY CHECK #2: NULL Analysis
Check for missing values in critical columns.

In [ ]:
-- NULL check on FACT_INTERNET_SALES critical columns
SELECT 
    COUNT(*) AS total_rows,
    COUNT(CASE WHEN SALESAMOUNT IS NULL THEN 1 END) AS null_salesamount,
    COUNT(CASE WHEN PRODUCTKEY IS NULL THEN 1 END) AS null_productkey,
    COUNT(CASE WHEN CUSTOMERKEY IS NULL THEN 1 END) AS null_customerkey,
    COUNT(CASE WHEN ORDERDATE IS NULL THEN 1 END) AS null_orderdate,
    COUNT(CASE WHEN CARRIERTRACKINGNUMBER IS NULL OR CARRIERTRACKINGNUMBER = 'NULL' THEN 1 END) AS null_tracking,
    COUNT(CASE WHEN CUSTOMERPONUMBER IS NULL OR CUSTOMERPONUMBER = 'NULL' THEN 1 END) AS null_po_number
FROM FACT_INTERNET_SALES;

In [ ]:
-- NULL check on DIM_CUSTOMER
SELECT 
    COUNT(*) AS total_rows,
    COUNT(CASE WHEN TITLE IS NULL OR TITLE = 'NULL' THEN 1 END) AS null_title,
    COUNT(CASE WHEN MIDDLENAME IS NULL OR MIDDLENAME = 'NULL' THEN 1 END) AS null_middlename,
    COUNT(CASE WHEN SUFFIX IS NULL OR SUFFIX = 'NULL' THEN 1 END) AS null_suffix,
    COUNT(CASE WHEN ADDRESSLINE2 IS NULL OR ADDRESSLINE2 = 'NULL' THEN 1 END) AS null_address2,
    COUNT(CASE WHEN YEARLYINCOME IS NULL THEN 1 END) AS null_income,
    COUNT(CASE WHEN GENDER IS NULL OR GENDER = 'NULL' THEN 1 END) AS null_gender
FROM DIM_CUSTOMER;

### DATA QUALITY CHECK #3: Date range validation

In [ ]:
-- Verify date ranges make sense
SELECT 
    MIN(ORDERDATE) AS first_order,
    MAX(ORDERDATE) AS last_order,
    MIN(SHIPDATE) AS first_ship,
    MAX(SHIPDATE) AS last_ship,
    AVG(DATEDIFF('day', ORDERDATE, SHIPDATE)) AS avg_shipping_days,
    MAX(DATEDIFF('day', ORDERDATE, SHIPDATE)) AS max_shipping_days
FROM FACT_INTERNET_SALES;

### Explore DIM_GEOGRAPHY and territory mapping

In [ ]:
# Load geography using pandas on Snowflake
geo_mdf = spd.read_snowflake('DIM_GEOGRAPHY')

print("=== Countries ===")
print(geo_mdf['ENGLISHCOUNTRYREGIONNAME'].value_counts())

print("\n=== Territory Keys ===")
print(geo_mdf['SALESTERRITORYKEY'].value_counts())

print(f"\nTotal geography records: {len(geo_mdf)}")

### Explore FACT_BUDGET

In [ ]:
# Load budget data
budget_mdf = spd.read_snowflake('FACT_BUDGET')

print("=== Budget Data ===")
print(budget_mdf)

print(f"\nTotal budget (all months): ${budget_mdf['BUDGET'].sum():,.0f}")

### Quick analysis: Sales by territory (using Snowpark DataFrame API)

In [ ]:
# Sales by territory using Snowpark DataFrame API
# (same pattern as DE_100 notebook's product_sentiment calculation)
territory_sales = fact_sales_sdf.group_by('SALESTERRITORYKEY') \
    .agg(
        F.sum('SALESAMOUNT').alias('TOTAL_REVENUE'),
        F.count('*').alias('ORDER_COUNT'),
        F.round(F.avg('SALESAMOUNT'), 2).alias('AVG_ORDER_VALUE')
    ) \
    .sort(F.col('TOTAL_REVENUE').desc())

print("\n=== Sales by Territory ===")
territory_sales.show()

### Step 1 Summary: Data Quality Findings

**What we found:**

| Issue | Impact | Resolution (Step 2) |
|-------|--------|---------------------|
| Product keys 310-351 in sales, but DIM_PRODUCT only has 1-100 | Product joins return NULL | Create enriched product lookup |
| ~87 customer keys unmatched between sales and DIM_CUSTOMER | Partial customer demographics | LEFT JOIN + territory fallback |
| 100% NULL in CARRIERTRACKINGNUMBER and CUSTOMERPONUMBER | Expected for internet sales | Document, exclude from analysis |
| TITLE, SUFFIX 100% NULL in DIM_CUSTOMER | Non-critical fields | Drop from curated table |

**Next step:** Run `Step2_Transformation.ipynb` to clean, enrich, and join all tables into a curated star schema.

In [ ]:
# Print summary of all findings
print("=" * 60)
print("STEP 1 COMPLETE: INGESTION & EXPLORATION")
print("=" * 60)
print(f"")
print(f"Database: SALES_ANALYTICS_2025")
print(f"Schema:   RAW_DATA (8 tables loaded)")
print(f"Schema:   CURATED  (created, empty — ready for Step 2)")
print(f"")
print(f"Tables explored:")
print(f"  ✓ FACT_INTERNET_SALES  — {fact_sales_sdf.count()} orders")
print(f"  ✓ DIM_CUSTOMER         — {len(customer_mdf)} customers")
print(f"  ✓ DIM_GEOGRAPHY        — {len(geo_mdf)} locations")
print(f"  ✓ DIM_PRODUCT          — {len(product_mdf)} products")
print(f"  ✓ DIM_PRODUCT_CATEGORY — 4 categories")
print(f"  ✓ DIM_PRODUCT_SUBCATEGORY — {len(subcategory_mdf)} subcategories")
print(f"  ✓ FACT_BUDGET          — {len(budget_mdf)} months")
print(f"")
print(f"Data quality issues identified: 4")
print(f"Resolution plan: Step2_Transformation.ipynb")
print(f"")
print(f"▶ NEXT: Open Step2_Transformation.ipynb")